# `protein_based_graph` — ranking molecules by binding-profile information

**Runs on the server; GPU-accelerated with a CPU fallback.** Loads only cached
embeddings + LORAX split indexes, so it is self-contained.

**Idea.** Our best cold-molecule (inductive) graph keeps message-passing edges from only
the top ~1 % of molecules by *raw coverage* (`q99` — 6 molecules). Coverage is a blunt
criterion: it ignores the **pos/neg composition** of a profile, ignores **which** proteins
it touches, and so cannot see that a small, well-balanced profile over otherwise-uncovered
proteins can carry more usable signal than a large all-negative one.

This notebook ranks molecules by several **information criteria of their binding profile**,
then asks: *how much does each ranking agree with the coverage ranking we use now?* — and,
concretely, *does an information-optimal set of the same size connect more of the protein
graph than the coverage-selected set?* The answer motivates the protein-based graph
architecture we will design next.

> Step 1 of the `protein_based_graph` line. Verdict-first, like the other `alternatives/`
> notebooks: nothing here trains a model — it is a data-geometry study of the MP graph.


## Setup — repo root, device (GPU→CPU), regime

In [ ]:
import sys, pathlib, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau

# resolve repo root by walking up to pyproject.toml (works from any depth)
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from orbind import lorax as L

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"repo root: {ROOT}")
print(f"device:    {DEVICE}")

# The coverage quality filter is computed per-regime on that regime's train edges.
# q99 wins on the cold-molecule regime, so we study it here (fold 1 / seed 42 — the
# per-molecule coverage of NON-held-out molecules is split-independent, so the
# ranking is representative of the whole regime).
REGIME, FOLD, SEED = "inductive_molecule", 1, 42

# palette (validated categorical set from the dataviz skill)
C_COV, C_GREEDY, C_RANDOM, C_COMP = "#808080", "#2a78d6", "#eda100", "#eb6834"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False})

## Build the profiles

A molecule's **profile** is the set of `(protein, label)` measurements it appears in on the
MP graph (train edges only — exactly the input the `q##` filter sees). We record:

- `coverage` — total measurements. **This is what `quality_mol_mask` ranks on today.**
- `n_pos / n_neg` — label composition.
- `distinct_prot` — number of distinct proteins the molecule connects (edge multiplicity
  collapsed): its "channel set" in the bipartite graph.

In [ ]:
esm, chem = L.load_embeddings()
Xm, Xp, splits = L.build(REGIME, FOLD, esm, chem, seed=SEED)
n_mol, n_prot = Xm.shape[0], Xp.shape[0]
pos, neg = splits["train"]["pos"], splits["train"]["neg"]      # (E,2): [mol, prot]

cov  = np.bincount(np.concatenate([pos[:, 0], neg[:, 0]]), minlength=n_mol)  # == filter input
npos = np.bincount(pos[:, 0], minlength=n_mol)
nneg = np.bincount(neg[:, 0], minlength=n_mol)

all_edges = np.concatenate([pos, neg], axis=0)
prof = [set() for _ in range(n_mol)]
for m, p in all_edges:
    prof[int(m)].add(int(p))
ndist  = np.array([len(s) for s in prof])
active = np.where(cov > 0)[0]                                   # rankable pool
print(f"{n_mol} molecules / {n_prot} proteins; {len(active)} molecules carry >=1 MP edge")

## Static per-molecule information criteria

Each is a different answer to "how much does this profile tell us?", targeting one of the
weaknesses of raw coverage.

| criterion | formula | what it rewards | fixes |
|---|---|---|---|
| `coverage` | `n` | more measurements | — *(current filter)* |
| `balance_bits` | `H₂(pos_frac)` | balanced pos/neg (per-obs bits) | all-neg profiles are worthless |
| `entropy_bits` | `n · H₂(pos_frac)` | total label information | size **and** balance |
| `disc_pairs` | `n_pos · n_neg` | protein-pairs the molecule *separates* | pure discrimination |
| `idf_coverage` | `Σ_p log₂(M / df_p)` | reaching **rarely-covered** proteins | non-redundant reach |
| `composite` | `balance_bits · idf_coverage` | rare reach **gated by** discrimination | all three at once |

`df_p` = number of distinct molecules that touch protein `p`; `M` = active molecules.
Note `balance_bits` (and `disc_pairs`) go to **0** for a constant-label profile — for the
signed message passing that is the honest verdict: a profile with no within-molecule
contrast provides no discriminative structure, only weak "these all behave alike" grouping.

In [ ]:
def h2(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

pos_frac     = np.divide(npos, np.maximum(cov, 1))
balance      = np.where(cov > 0, h2(pos_frac), 0.0)
entropy_bits = cov * balance
disc_pairs   = npos.astype(float) * nneg.astype(float)

df = np.zeros(n_prot)
for s in prof:
    for p in s:
        df[p] += 1
M_active = len(active)
idf      = np.where(df > 0, np.log2(M_active / np.maximum(df, 1)), 0.0)
idf_cov  = np.array([idf[list(s)].sum() if s else 0.0 for s in prof])
composite = balance * idf_cov
print("static criteria computed")

## Greedy protein-pair coverage (joint, redundancy-aware) — GPU

The static scores judge each molecule alone. But two molecules with near-identical profiles
are jointly redundant. The **graph** cares about which protein-*pairs* get connected (a
molecule links every pair of proteins in its profile). So we rank molecules by a submodular
set objective: repeatedly pick the molecule that **newly** connects the most protein-pairs.

For a candidate molecule with 0/1 protein indicator `v` and a running boolean "already
covered pairs" matrix `C`, its marginal gain is `‖v‖² − vᵀ C v` (ordered pairs). This is one
`(remaining × P) @ (P × P)` matmul per step — the GPU workload, trivial on `cuda`, ~15 s on CPU.

A redundant molecule scores low automatically; a **small but unique** profile can rank high —
exactly the case raw coverage cannot represent.

In [ ]:
V = torch.zeros(n_mol, n_prot, device=DEVICE)
for m in active:
    V[m, torch.tensor(sorted(prof[m]), device=DEVICE)] = 1.0
n_touch = V.sum(1)

def greedy_pair_cover():
    C = torch.zeros(n_prot, n_prot, device=DEVICE)
    seen = torch.zeros(n_prot, device=DEVICE)
    remaining = set(int(m) for m in active)
    order, gains, cpairs, cprot = [], [], [], []
    while remaining:
        rem  = torch.tensor(sorted(remaining), device=DEVICE)
        Vr   = V[rem]
        gain = n_touch[rem] ** 2 - (Vr @ C * Vr).sum(1)     # newly covered ordered pairs
        j    = int(torch.argmax(gain).item())
        m    = int(rem[j].item()); v = V[m]
        C    = torch.maximum(C, torch.outer(v, v))
        seen = torch.maximum(seen, v)
        order.append(m); gains.append(float(gain[j].item()))
        cpairs.append(float(((C.sum() - C.diagonal().sum()) / 2).item()))
        cprot.append(int(seen.sum().item()))
        remaining.discard(m)
    return order, gains, np.array(cpairs), np.array(cprot)

t0 = time.time()
g_order, g_gains, g_cum_pairs, g_cum_prot = greedy_pair_cover()
greedy_rank = {m: i for i, m in enumerate(g_order)}
print(f"greedy done in {time.time()-t0:.1f}s on {DEVICE}; first 6 picks: {g_order[:6]}")

## Assemble the ranking table

In [ ]:
CRIT = ["coverage", "balance_bits", "entropy_bits", "disc_pairs", "idf_coverage", "composite"]
d = pd.DataFrame({
    "mol": np.arange(n_mol), "coverage": cov, "n_pos": npos, "n_neg": nneg,
    "distinct_prot": ndist, "pos_frac": pos_frac, "balance_bits": balance,
    "entropy_bits": entropy_bits, "disc_pairs": disc_pairs,
    "idf_coverage": idf_cov, "composite": composite,
}).iloc[active].reset_index(drop=True)
d["greedy_pair_cover"] = d["mol"].map(greedy_rank)           # 0 = best
for c in CRIT:
    d[c + "_rank"] = d[c].rank(ascending=False, method="min")
d["greedy_rank"] = d["greedy_pair_cover"] + 1
d.sort_values("greedy_pair_cover").head(10)

## Train `gnn signed` per (selection criterion × quantile × seed) + boost references

For each selection criterion, keep the MP edges of only its **top-K molecules**, train **our
standard `gnn signed`**, and read cold-molecule test metrics off the unentangled XGBoost probe.
A no-graph **boost baseline** is computed alongside for comparison:

- **`boost (full)`** — raw XGBoost on `[raw mol ‖ raw prot]`, trained on **all** train pairs.
  Quantile-independent → a flat reference; this is the real number the graph must beat.
  *(A quantile-restricted boost is left for later.)*

**Design:** `K` per quantile from the coverage filter (same set sizes `q50/80/85/90/95/99` use
today); each criterion keeps its own top-K. **3 seeds** (42/43/44) — in the inductive regime a
seed defines the cold split, so profiles/criteria/greedy order/`K` are recomputed per seed.
Standard config: `signed`, inductive, ChemBERTa, 900 epochs, lr 3e-3, grad-clip 1.0, hidden
256, dropout 0.3, last-epoch probe. **Incremental cache** on `(criterion, quantile, seed)`.

> Two figures below: absolute values (mean ± 95% CI) and the same as **mean rank** over methods.

In [ ]:
from orbind import hetero as H
from orbind.baselines import train_boost
from orbind.dataset import metrics

EPOCHS, LR, GRAD_CLIP, HIDDEN, DROPOUT = 900, 3e-3, 1.0, 256, 0.3
QUANTILES = [50, 80, 85, 90, 95, 99]
BOOST_QUANTILES = [35, 50, 65, 80, 85, 90, 95, 99]   # boost run adds 35 & 65
SEEDS = [42, 43, 44, 45, 46]                        # boost seed = seed + 1000
dev = torch.device(DEVICE)

def train_gnn_signed(keep_mols, splits_s, seed, boost_seed, epochs=EPOCHS):
    '''Standard signed bipartite GNN, inductive, raw-ChemBERTa probe. MP edges are
    restricted to keep_mols; supervision/probe use splits_s train edges.'''
    torch.manual_seed(seed); np.random.seed(seed)
    if dev.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    x_dict = {H.MOL: Xm.clone(), H.PROT: Xp.clone()}      # raw features (no PCA)
    gnn_train = probe_train = splits_s["train"]
    keep = set(int(m) for m in keep_mols)
    filt = lambda a: a[np.array([int(m) in keep for m in a[:, 0]], dtype=bool)] if len(a) else a
    mp_pos = torch.tensor(filt(gnn_train["pos"]).T, dtype=torch.long)
    mp_neg = torch.tensor(filt(gnn_train["neg"]).T, dtype=torch.long)
    eidx = H.edge_index_dict(mp_pos, mp_neg, mode="signed")
    sup = {k: H.sup_edges(v) for k, v in
           {"train": gnn_train, "probe_train": probe_train,
            "val": splits_s["val"], "test": splits_s["test"]}.items()}

    x_dict = {k: v.to(dev) for k, v in x_dict.items()}
    eidx = {k: v.to(dev) for k, v in eidx.items()}
    sup = {k: (i.to(dev), y.to(dev)) for k, (i, y) in sup.items()}

    model = H.HeteroLink(hidden=HIDDEN, dropout=DROPOUT, mp_mode="signed").to(dev)
    with torch.no_grad():
        model.encode(x_dict, eidx)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    tr_idx, tr_y = sup["train"]
    pw = ((tr_y == 0).sum().float() / (tr_y == 1).sum().float().clamp_min(1.0)).reshape(1).to(dev)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pw)

    for ep in range(epochs):
        model.train(); opt.zero_grad()
        z = model.encode(x_dict, eidx)
        loss = loss_fn(model.decode(z, tr_idx), tr_y)
        loss.backward()
        if GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

    model.eval()
    with torch.no_grad():
        z = model.encode(x_dict, eidx)
    Xm_probe = x_dict[H.MOL].detach().cpu().numpy()
    Zp = z[H.PROT].detach().cpu().numpy()
    def feats(idx):
        ii = idx.detach().cpu().numpy()
        return np.concatenate([Xm_probe[ii[0]], Zp[ii[1]]], axis=1)
    ytr = sup["probe_train"][1].detach().cpu().numpy()
    yte = sup["test"][1].detach().cpu().numpy()
    sc = train_boost(feats(sup["probe_train"][0]), ytr, feats(sup["test"][0]), seed=boost_seed)
    return metrics(yte, sc)

def boost_no_graph(splits_s, keep_mols, seed, boost_seed):
    '''Raw XGBoost on [raw ChemBERTa mol || raw ESM prot], no graph. If keep_mols is
    given, training pairs are restricted to those molecules (the boost q-filter);
    test is always the full cold test.'''
    Xm_np, Xp_np = Xm.numpy(), Xp.numpy()
    tr_idx, tr_y = H.sup_edges(splits_s["train"]); tr_idx = tr_idx.numpy(); tr_y = tr_y.numpy()
    te_idx, te_y = H.sup_edges(splits_s["test"]);  te_idx = te_idx.numpy(); te_y = te_y.numpy()
    if keep_mols is not None:
        keep = set(int(m) for m in keep_mols)
        m = np.array([int(x) in keep for x in tr_idx[0]], dtype=bool)
        tr_idx, tr_y = tr_idx[:, m], tr_y[m]
    feats = lambda idx: np.concatenate([Xm_np[idx[0]], Xp_np[idx[1]]], axis=1)
    sc = train_boost(feats(tr_idx), tr_y, feats(te_idx), seed=boost_seed)
    return metrics(te_y, sc)

print("train_gnn_signed + boost_no_graph ready on", DEVICE)

In [ ]:
# per-seed criteria + selection (mirrors the exposition cells above, generalized to any seed)
def compute_scores(splits_s):
    pos_s, neg_s = splits_s["train"]["pos"], splits_s["train"]["neg"]
    cov_s  = np.bincount(np.concatenate([pos_s[:, 0], neg_s[:, 0]]), minlength=n_mol)
    npos_s = np.bincount(pos_s[:, 0], minlength=n_mol)
    nneg_s = np.bincount(neg_s[:, 0], minlength=n_mol)
    prof_s = [set() for _ in range(n_mol)]
    for m, p in np.concatenate([pos_s, neg_s], axis=0):
        prof_s[int(m)].add(int(p))
    active_s = np.where(cov_s > 0)[0]
    balance_s = np.where(cov_s > 0, h2(np.divide(npos_s, np.maximum(cov_s, 1))), 0.0)
    df_s = np.zeros(n_prot)
    for s in prof_s:
        for p in s:
            df_s[p] += 1
    idf_s = np.where(df_s > 0, np.log2(len(active_s) / np.maximum(df_s, 1)), 0.0)
    idf_cov_s = np.array([idf_s[list(s)].sum() if s else 0.0 for s in prof_s])
    SCORE = {"coverage": cov_s, "balance_bits": balance_s,
             "entropy_bits": cov_s * balance_s,
             "disc_pairs": npos_s.astype(float) * nneg_s.astype(float),
             "idf_coverage": idf_cov_s, "composite": balance_s * idf_cov_s}
    V = torch.zeros(n_mol, n_prot, device=dev)
    for m in active_s:
        V[m, torch.tensor(sorted(prof_s[m]), device=dev)] = 1.0
    n_touch = V.sum(1); C = torch.zeros(n_prot, n_prot, device=dev)
    rem = set(int(m) for m in active_s); order = []
    while rem:
        r = torch.tensor(sorted(rem), device=dev); Vr = V[r]
        gain = n_touch[r] ** 2 - (Vr @ C * Vr).sum(1)
        j = int(torch.argmax(gain).item()); m = int(r[j].item()); v = V[m]
        C = torch.maximum(C, torch.outer(v, v)); order.append(m); rem.discard(m)
    Kq = {q: int(H.quality_mol_mask(pos_s, neg_s, n_mol, q / 100).sum()) for q in sorted(set(QUANTILES) | set(BOOST_QUANTILES))}
    return {"splits": splits_s, "cov": cov_s, "SCORE": SCORE, "g_order": order, "Kq": Kq}

CRITERIA = ["coverage", "balance_bits", "entropy_bits", "disc_pairs",
            "idf_coverage", "composite", "greedy_pair_cover"]

def select(crit, sc, K):
    if crit == "greedy_pair_cover":
        return sc["g_order"][:K]
    return [int(m) for m in np.argsort(-sc["SCORE"][crit]) if sc["cov"][m] > 0][:K]

per_seed = {s: compute_scores(L.build(REGIME, FOLD, esm, chem, seed=s)[2]) for s in SEEDS}
print("K per (seed, quantile):")
for s in SEEDS:
    print(f"  seed {s}: {per_seed[s]['Kq']}")

In [ ]:
# incremental run + cache (criterion x quantile x seed); GNN criteria + 2 boost baselines
CACHE = ROOT / "results/graph/full_full/v6/protein_based_graph"
CACHE.mkdir(parents=True, exist_ok=True)
CSV = CACHE / f"metrics_{REGIME}.csv"
# robust load: ingest the canonical file AND any legacy per-seed CSVs, so runs
# written before the cache filename was unified are still picked up.
_frames = []
for _f in sorted(CACHE.glob(f"metrics_{REGIME}*.csv")):
    if _f.name.endswith(".tmp.csv"):
        continue
    _df = pd.read_csv(_f)
    if {"criterion", "quantile", "seed"}.issubset(_df.columns) and len(_df):
        _frames.append(_df)
done = (pd.concat(_frames, ignore_index=True)
        .drop_duplicates(["criterion", "quantile", "seed"], keep="last")
        if _frames else pd.DataFrame(columns=["criterion", "quantile", "seed"]))
rows = done.to_dict("records")
if len(done):
    _cr = sorted(set(done["criterion"])); _q = sorted(set(done["quantile"])); _sd = sorted(set(done["seed"]))
    print(f"loaded {len(done)} cached results from {len(_frames)} file(s) | "
          f"criteria={_cr} q={_q} seeds={_sd}")

def already(crit, q, seed):
    return len(done) and ((done["criterion"] == crit) & (done["quantile"] == q) & (done["seed"] == seed)).any()

def save():
    tmp = CSV.with_suffix(".tmp.csv")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    tmp.replace(CSV)

def add_row(crit, q, K, seed, m):
    global done
    rows.append({"criterion": crit, "quantile": q, "K": K, "seed": seed, **m})
    done = pd.DataFrame(rows)
    save()

t0 = time.time()
for seed in SEEDS:
    sc = per_seed[seed]; splits_s = sc["splits"]; bseed = seed + 1000
    # --- boost (full): quantile-independent, written once per quantile so it ranks everywhere
    if not all(already("boost_full", q, seed) for q in QUANTILES):
        mf = boost_no_graph(splits_s, None, seed, bseed)
        for q in QUANTILES:
            if not already("boost_full", q, seed):
                add_row("boost_full", q, -1, seed, mf)
        print(f"seed {seed} boost_full         : AUROC={mf['AUROC']:.3f} AUPRC={mf['AUPRC']:.3f}")
    for q in QUANTILES:
        K = sc["Kq"][q]
        # --- GNN per selection criterion
        for crit in CRITERIA:
            if already(crit, q, seed):
                continue
            m = train_gnn_signed(select(crit, sc, K), splits_s, seed=seed, boost_seed=bseed)
            add_row(crit, q, K, seed, m)
            print(f"seed {seed} q{q:2d} {crit:18s} K={K:3d}: "
                  f"AUROC={m['AUROC']:.3f} AUPRC={m['AUPRC']:.3f} "
                  f"MCC={m['MCC']:.3f} F1={m['F1']:.3f}  [{time.time()-t0:.0f}s]")
res = pd.read_csv(CSV)
n_methods = len(CRITERIA) + 1
print(f"\n{len(res)}/{n_methods*len(QUANTILES)*len(SEEDS)} cells present -> {CSV.relative_to(ROOT)}")

## Results — absolute values (one panel per metric, mean ± 95% CI over seeds)

Each panel is a metric; each curve connects a method's mean cold-molecule score across
quantiles, whiskers = 95% t-CI over the 3 seeds. The black line is the no-graph
`boost (full)` (flat — the number to beat).

In [ ]:
from scipy.stats import t as _t
res = pd.read_csv(CSV)
METR = ["AUROC", "AUPRC", "MCC", "F1"]

# 7 GNN criteria = categorical colors; boost = neutral reference line
COL = {"coverage": "#2a78d6", "balance_bits": "#eb6834", "entropy_bits": "#1baf7a",
       "disc_pairs": "#eda100", "idf_coverage": "#e87ba4", "composite": "#008300",
       "greedy_pair_cover": "#4a3aa7",
       "boost_full": "#0b0b0b"}
LABEL = {"boost_full": "boost (full)"}
ORDER = CRITERIA + ["boost_full"]

def sty(c):
    if c == "boost_full":
        return dict(ls="-", lw=2.6, marker="s", zorder=6)
    return dict(ls="-", lw=2.0, marker="o", zorder=3)

def mean_ci(vals):
    v = np.asarray(vals, float); n = len(v)
    if n < 2:
        return (v.mean() if n else np.nan), 0.0
    return v.mean(), _t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)

# small horizontal dodge per method so overlapping whiskers separate
_dodge = {c: (i - (len(ORDER) - 1) / 2) * 0.55 for i, c in enumerate(ORDER)}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    for c in ORDER:
        sub = res[res.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means, errs = zip(*[mean_ci(sub[sub["quantile"] == q][metric].values) for q in qs])
        xd = [q + _dodge[c] for q in qs]
        st = sty(c)
        ax.plot(xd, means, ms=5, color=COL[c], label=LABEL.get(c, c), **st)
        ax.errorbar(xd, means, yerr=errs, fmt="none", ecolor=COL[c],
                    elinewidth=1.1, capsize=2, alpha=0.3, zorder=st["zorder"] - 1)
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel(metric); ax.set_xticks(QUANTILES)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"gnn signed · inductive · mean of seeds {SEEDS} (95% CI) — value vs quantile", y=1.06)
fig.tight_layout(); plt.show()

## Results — mean rank (same panels, ranked instead of absolute)

Same layout, but each panel ranks all methods against each other **within every
`(quantile, seed)`** (1 = best on that metric), then averages the rank over the 3 seeds.
The y-axis is inverted so higher = better; whiskers = 95% t-CI of the rank across seeds.
Rank cancels the large per-seed level shifts of the cold regime, so it reads the *ordering*
of methods more robustly than the absolute values.

In [ ]:
res = pd.read_csv(CSV)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    r = res.copy()
    # rank methods within each (quantile, seed); higher metric = better = rank 1
    r["rk"] = r.groupby(["quantile", "seed"])[metric].rank(ascending=False, method="average")
    for c in ORDER:
        sub = r[r.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means = [sub[sub["quantile"] == q]["rk"].mean() for q in qs]
        ax.plot(qs, means, ms=5, color=COL[c], label=LABEL.get(c, c), **sty(c))
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel("mean rank (1 = best)"); ax.set_xticks(QUANTILES)
    ax.invert_yaxis()
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"gnn signed · inductive · mean of seeds {SEEDS} — mean rank vs quantile", y=1.06)
fig.tight_layout(); plt.show()

## Honest boost baselines on the same quantile selections

The GNN figures above answer "does the graph help?" — but only against a single full-data
boost. The fair control is **the same molecule selection fed to plain boosting instead of the
graph**: for each criterion × quantile, restrict the raw XGBoost `[raw mol ‖ raw prot]`
training pairs to that criterion's top-K molecules (no graph at all), and also keep the
quantile-independent `boost (full)`.

Comparing these curves to the GNN curves isolates the graph's contribution: if a GNN criterion
beats its boost twin here, the *message passing* — not just the molecule subset — is doing the
work. Separate incremental cache: `metrics_boost_{REGIME}.csv`.

In [ ]:
# honest no-graph boost on each (criterion, quantile, seed) selection + boost_full
BCSV = CACHE / f"metrics_boost_{REGIME}.csv"
_bframes = []
for _f in sorted(CACHE.glob(f"metrics_boost_{REGIME}*.csv")):
    if _f.name.endswith(".tmp.csv"):
        continue
    _df = pd.read_csv(_f)
    if {"criterion", "quantile", "seed"}.issubset(_df.columns) and len(_df):
        _bframes.append(_df)
bdone = (pd.concat(_bframes, ignore_index=True)
         .drop_duplicates(["criterion", "quantile", "seed"], keep="last")
         if _bframes else pd.DataFrame(columns=["criterion", "quantile", "seed"]))
brows = bdone.to_dict("records")
if len(bdone):
    print(f"loaded {len(bdone)} cached boost results")

def balready(crit, q, seed):
    return len(bdone) and ((bdone["criterion"] == crit) & (bdone["quantile"] == q) & (bdone["seed"] == seed)).any()

def bsave():
    tmp = BCSV.with_suffix(".tmp.csv")
    pd.DataFrame(brows).to_csv(tmp, index=False)
    tmp.replace(BCSV)

def badd(crit, q, K, seed, m):
    global bdone
    brows.append({"criterion": crit, "quantile": q, "K": K, "seed": seed, **m})
    bdone = pd.DataFrame(brows)
    bsave()

t0 = time.time()
for seed in SEEDS:
    sc = per_seed[seed]; splits_s = sc["splits"]; bs = seed + 1000
    if not all(balready("boost_full", q, seed) for q in BOOST_QUANTILES):
        mf = boost_no_graph(splits_s, None, seed, bs)
        for q in BOOST_QUANTILES:
            if not balready("boost_full", q, seed):
                badd("boost_full", q, -1, seed, mf)
        print(f"seed {seed} boost_full         : AUROC={mf['AUROC']:.3f} AUPRC={mf['AUPRC']:.3f}")
    for q in BOOST_QUANTILES:
        K = sc["Kq"][q]
        for crit in CRITERIA:
            if balready(crit, q, seed):
                continue
            m = boost_no_graph(splits_s, select(crit, sc, K), seed, bs)
            badd(crit, q, K, seed, m)
            print(f"seed {seed} q{q:2d} boost/{crit:18s} K={K:3d}: "
                  f"AUROC={m['AUROC']:.3f} AUPRC={m['AUPRC']:.3f} "
                  f"MCC={m['MCC']:.3f} F1={m['F1']:.3f}  [{time.time()-t0:.0f}s]")
bres = pd.read_csv(BCSV)
print(f"\n{len(bres)}/{(len(CRITERIA)+1)*len(BOOST_QUANTILES)*len(SEEDS)} boost cells present -> {BCSV.relative_to(ROOT)}")

### Boost baselines — absolute values (mean ± 95% CI over seeds)

Same layout and styling as the GNN absolute figure, but every curve is a **no-graph** boost:
`coverage`/`balance_bits`/… = boost trained only on that criterion's top-K molecules;
`boost (full)` = boost on all molecules.

In [ ]:
bres = pd.read_csv(BCSV)
_dodge = {c: (i - (len(ORDER) - 1) / 2) * 0.55 for i, c in enumerate(ORDER)}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    for c in ORDER:
        sub = bres[bres.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means, errs = zip(*[mean_ci(sub[sub["quantile"] == q][metric].values) for q in qs])
        xd = [q + _dodge[c] for q in qs]
        st = sty(c)
        ax.plot(xd, means, ms=5, color=COL[c], label=LABEL.get(c, c), **st)
        ax.errorbar(xd, means, yerr=errs, fmt="none", ecolor=COL[c],
                    elinewidth=1.1, capsize=2, alpha=0.3, zorder=st["zorder"] - 1)
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel(metric); ax.set_xticks(BOOST_QUANTILES)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"boost (no graph) on quantile selections · inductive · mean of seeds {SEEDS} (95% CI) — value vs quantile", y=1.06)
fig.tight_layout(); plt.show()

### Boost baselines — mean rank (same panels, ranked, no whiskers)

In [ ]:
bres = pd.read_csv(BCSV)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    r = bres.copy()
    r["rk"] = r.groupby(["quantile", "seed"])[metric].rank(ascending=False, method="average")
    for c in ORDER:
        sub = r[r.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means = [sub[sub["quantile"] == q]["rk"].mean() for q in qs]
        ax.plot(qs, means, ms=5, color=COL[c], label=LABEL.get(c, c), **sty(c))
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel("mean rank (1 = best)"); ax.set_xticks(BOOST_QUANTILES)
    ax.invert_yaxis()
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"boost (no graph) on quantile selections · inductive · mean of seeds {SEEDS} — mean rank vs quantile", y=1.06)
fig.tight_layout(); plt.show()

## Bagged GNN — matching the ensembler's `cls` protocol exactly

The single-model GNN above underperforms the ensembler's `GNN cls+mol` because that source is a
**bag of `n_models=5` independently-seeded GNNs whose protein embeddings are concatenated**
(`orbind.gnn_extractor.GnnSignedExtractor`, `emit="prot"` gives `n_models x hidden = 1280`
protein features) before a single boost head. This section reproduces that: for each criterion x
quantile x seed, train `N_MODELS` GNNs (seed `s + 5000*m`, else identical: signed, q-filter,
ESM-1b, ChemBERTa nodes, lr 3e-3, 900 epochs, last checkpoint, grad-clip 1.0), concatenate their
protein embeddings, and fit one XGBoost on `[raw ChemBERTa mol || 1280-d bagged protein]`.

Same selection criteria, same two figures, separate incremental cache
`metrics_gnnbag_{REGIME}.csv`.

> Cost is `N_MODELS x` the single-model run (7 criteria x 6 quantiles x 5 seeds x 5 models x
> 900 epochs) - GPU only; the cache lets you fill it in chunks.

In [ ]:
N_MODELS, SEED_OFFSET = 5, 5000    # ensembler default: 5-model bag, seed s + 5000*m

def _encode_prot(keep_mols, splits_s, seed, epochs=EPOCHS):
    """Train ONE signed GNN (MP edges restricted to keep_mols) and return its
    graph-enriched protein embedding for every protein node (n_prot, hidden)."""
    torch.manual_seed(seed); np.random.seed(seed)
    if dev.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    x_dict = {H.MOL: Xm.clone(), H.PROT: Xp.clone()}
    gnn_train = splits_s["train"]
    keep = set(int(m) for m in keep_mols)
    filt = lambda a: a[np.array([int(m) in keep for m in a[:, 0]], dtype=bool)] if len(a) else a
    mp_pos = torch.tensor(filt(gnn_train["pos"]).T, dtype=torch.long)
    mp_neg = torch.tensor(filt(gnn_train["neg"]).T, dtype=torch.long)
    eidx = H.edge_index_dict(mp_pos, mp_neg, mode="signed")
    tr_idx, tr_y = H.sup_edges(gnn_train)
    x_dict = {k: v.to(dev) for k, v in x_dict.items()}
    eidx = {k: v.to(dev) for k, v in eidx.items()}
    tr_idx, tr_y = tr_idx.to(dev), tr_y.to(dev)
    model = H.HeteroLink(hidden=HIDDEN, dropout=DROPOUT, mp_mode="signed").to(dev)
    with torch.no_grad():
        model.encode(x_dict, eidx)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    pw = ((tr_y == 0).sum().float() / (tr_y == 1).sum().float().clamp_min(1.0)).reshape(1).to(dev)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pw)
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = loss_fn(model.decode(model.encode(x_dict, eidx), tr_idx), tr_y)
        loss.backward()
        if GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
    model.eval()
    with torch.no_grad():
        z = model.encode(x_dict, eidx)
    return z[H.PROT].detach().cpu().numpy()

def train_gnn_bagged(keep_mols, splits_s, seed, boost_seed, n_models=N_MODELS, seed_offset=SEED_OFFSET):
    """Bag n_models GNNs (seed s + seed_offset*m), concatenate their protein
    embeddings, fit one XGBoost on [raw ChemBERTa mol || bagged protein]."""
    Zp = np.concatenate([_encode_prot(keep_mols, splits_s, seed + seed_offset * m)
                         for m in range(n_models)], axis=1)
    Xm_np = Xm.numpy()
    tr_idx, tr_y = H.sup_edges(splits_s["train"]); tr_idx = tr_idx.numpy(); tr_y = tr_y.numpy()
    te_idx, te_y = H.sup_edges(splits_s["test"]);  te_idx = te_idx.numpy(); te_y = te_y.numpy()
    feats = lambda idx: np.concatenate([Xm_np[idx[0]], Zp[idx[1]]], axis=1)
    sc = train_boost(feats(tr_idx), tr_y, feats(te_idx), seed=boost_seed)
    return metrics(te_y, sc)

print(f"bagged GNN ready: N_MODELS={N_MODELS} on {DEVICE}")

In [ ]:
# bagged-GNN run (criterion x quantile x seed) + boost_full; separate incremental cache
GBCSV = CACHE / f"metrics_gnnbag_{REGIME}.csv"
_gframes = []
for _f in sorted(CACHE.glob(f"metrics_gnnbag_{REGIME}*.csv")):
    if _f.name.endswith(".tmp.csv"):
        continue
    _df = pd.read_csv(_f)
    if {"criterion", "quantile", "seed"}.issubset(_df.columns) and len(_df):
        _gframes.append(_df)
gbdone = (pd.concat(_gframes, ignore_index=True)
          .drop_duplicates(["criterion", "quantile", "seed"], keep="last")
          if _gframes else pd.DataFrame(columns=["criterion", "quantile", "seed"]))
gbrows = gbdone.to_dict("records")
if len(gbdone):
    print(f"loaded {len(gbdone)} cached bagged-GNN results")

def gbalready(crit, q, seed):
    return len(gbdone) and ((gbdone["criterion"] == crit) & (gbdone["quantile"] == q) & (gbdone["seed"] == seed)).any()

def gbsave():
    tmp = GBCSV.with_suffix(".tmp.csv")
    pd.DataFrame(gbrows).to_csv(tmp, index=False)
    tmp.replace(GBCSV)

def gbadd(crit, q, K, seed, m):
    global gbdone
    gbrows.append({"criterion": crit, "quantile": q, "K": K, "seed": seed, "n_models": N_MODELS, **m})
    gbdone = pd.DataFrame(gbrows)
    gbsave()

t0 = time.time()
for seed in SEEDS:
    sc = per_seed[seed]; splits_s = sc["splits"]; bs = seed + 1000
    if not all(gbalready("boost_full", q, seed) for q in QUANTILES):
        mf = boost_no_graph(splits_s, None, seed, bs)
        for q in QUANTILES:
            if not gbalready("boost_full", q, seed):
                gbadd("boost_full", q, -1, seed, mf)
        print(f"seed {seed} boost_full         : AUROC={mf['AUROC']:.3f} AUPRC={mf['AUPRC']:.3f}")
    for q in QUANTILES:
        K = sc["Kq"][q]
        for crit in CRITERIA:
            if gbalready(crit, q, seed):
                continue
            m = train_gnn_bagged(select(crit, sc, K), splits_s, seed=seed, boost_seed=bs)
            gbadd(crit, q, K, seed, m)
            print(f"seed {seed} q{q:2d} bag/{crit:18s} K={K:3d}: "
                  f"AUROC={m['AUROC']:.3f} AUPRC={m['AUPRC']:.3f} "
                  f"MCC={m['MCC']:.3f} F1={m['F1']:.3f}  [{time.time()-t0:.0f}s]")
gbres = pd.read_csv(GBCSV)
print(f"\n{len(gbres)}/{(len(CRITERIA)+1)*len(QUANTILES)*len(SEEDS)} bagged cells present -> {GBCSV.relative_to(ROOT)}")

### Bagged GNN — absolute values (mean +/- 95% CI over seeds)

Same layout/styling as the single-model GNN figure, but each graph curve is a **5-model bag**
(protocol-matched to the ensembler's `cls`). `boost (full)` is the flat no-graph reference.

In [ ]:
gbres = pd.read_csv(GBCSV)
_dodge = {c: (i - (len(ORDER) - 1) / 2) * 0.55 for i, c in enumerate(ORDER)}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    for c in ORDER:
        sub = gbres[gbres.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means, errs = zip(*[mean_ci(sub[sub["quantile"] == q][metric].values) for q in qs])
        xd = [q + _dodge[c] for q in qs]
        st = sty(c)
        ax.plot(xd, means, ms=5, color=COL[c], label=LABEL.get(c, c), **st)
        ax.errorbar(xd, means, yerr=errs, fmt="none", ecolor=COL[c],
                    elinewidth=1.1, capsize=2, alpha=0.3, zorder=st["zorder"] - 1)
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel(metric); ax.set_xticks(QUANTILES)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"bagged GNN (n_models={N_MODELS}) - inductive - mean of seeds {SEEDS} (95% CI) - value vs quantile", y=1.06)
fig.tight_layout(); plt.show()

### Bagged GNN — mean rank (same panels, ranked, no whiskers)

In [ ]:
gbres = pd.read_csv(GBCSV)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    r = gbres.copy()
    r["rk"] = r.groupby(["quantile", "seed"])[metric].rank(ascending=False, method="average")
    for c in ORDER:
        sub = r[r.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means = [sub[sub["quantile"] == q]["rk"].mean() for q in qs]
        ax.plot(qs, means, ms=5, color=COL[c], label=LABEL.get(c, c), **sty(c))
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel("mean rank (1 = best)"); ax.set_xticks(QUANTILES)
    ax.invert_yaxis()
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"bagged GNN (n_models={N_MODELS}) - inductive - mean of seeds {SEEDS} - mean rank vs quantile", y=1.06)
fig.tight_layout(); plt.show()